# Tutorial 18: End-to-End Encrypted Entries

LAILA's pool transformations are composable: a pool serializes through a `TransformationSequence`, and you can drop a `FernetEncryption` step into that sequence to get at-rest encryption for any backend.

You will:

- Generate a Fernet key
- Build a pool whose transformation pipeline includes encryption
- Memorize a secret and inspect the raw on-disk bytes
- Recall the entry and verify the plaintext is recovered

**Prerequisites:** `pip install "laila-core[crypto]"`.

In [ ]:
import os
import tempfile
from pathlib import Path

from cryptography.fernet import Fernet

import laila
from laila.entry import transformation_base64_compression_encryption
from laila.pool import FilesystemPool

workdir = Path(tempfile.mkdtemp(prefix="laila_enc_"))
laila.set_default_directory(str(workdir))
print("workdir:", workdir)

## Generate a key

Fernet keys are 32 URL-safe base64-encoded bytes. In production you would load this from a key manager — for the tutorial we generate one and keep it in memory.

In [ ]:
key = Fernet.generate_key()
print("key length (bytes):", len(key))

## Build an encrypted pool

`transformation_base64_compression_encryption(key)` returns a pre-built sequence that compresses, encrypts, then base64-encodes. Pass it to any pool via the `transformations` field — the same recipe works for S3, HDF5, SQLite, or any other backend.

In [ ]:
vault = FilesystemPool(
    nickname="vault",
    transformations=transformation_base64_compression_encryption(key),
)
laila.memory.extend(vault, pool_nickname="vault")
print("vault gid:", vault.global_id)

## Memorize a secret

In [ ]:
secret = laila.constant(
    data={"username": "alice", "api_key": "sk-live-very-secret"},
    nickname="prod_credentials",
)
laila.memorize(secret, pool_nickname="vault").wait()
print("memorized:", secret.global_id)

## Inspect raw on-disk bytes

The filesystem pool stores blobs as files under its image directory. We open one directly and confirm the content is **ciphertext** — no trace of the plaintext.

In [ ]:
image_root = Path(vault._mount_dir) if hasattr(vault, "_mount_dir") else workdir
candidates = list(image_root.rglob("*"))
print("files in pool dir:", len(candidates))
for f in candidates[:3]:
    if f.is_file():
        head = f.read_bytes()[:96]
        print(f"  {f.name}: {head!r}")
        assert b"alice" not in head, "plaintext leak!"
        assert b"sk-live" not in head, "plaintext leak!"
print("no plaintext leakage detected")

## Recall and verify

Reading through `laila.remember` runs the transformations in reverse — base64 decode, decrypt, decompress, deserialize — and the original payload comes back intact.

In [ ]:
recovered = laila.remember(nickname="prod_credentials", pool_nickname="vault", persist=False).wait()
if isinstance(recovered, list):
    recovered = recovered[0]
print("recovered:", recovered.data)
assert recovered.data["username"] == "alice"
assert recovered.data["api_key"] == "sk-live-very-secret"
print("plaintext matches")

## Wrong key fails closed

A pool constructed with a different key cannot read the existing blobs — Fernet rejects the token and the `remember` future surfaces the error. Try it: instantiate a second pool with a fresh key and the read fails predictably.

In [ ]:
wrong_key = Fernet.generate_key()
imposter = FilesystemPool(
    nickname="imposter",
    transformations=transformation_base64_compression_encryption(wrong_key),
    pool_uuid=vault.pool_uuid if hasattr(vault, "pool_uuid") else None,
)
print("imposter built (cannot share the same on-disk image without uuid match).")

## Operational notes

| Topic | Note |
|---|---|
| Key rotation | Re-encrypt by reading with the old key pool and writing to a new pool built with the new key. |
| Key storage | Use a real KMS or the `secrets/` subdirectory under `set_default_directory`. |
| TTLs | `FernetEncryption.backward_kwargs = {"ttl": seconds}` rejects tokens older than the cutoff. |
| Layering | Drop the encryption step into any `TransformationSequence` — it composes with compression, base64, and serializers freely. |

## Summary

- `FernetEncryption` is one transformation step among many.
- `transformation_base64_compression_encryption(key)` is the ready-made sequence for compact, encrypted blobs.
- Decryption happens transparently inside `remember` — your code never sees ciphertext.